# Hybrid: total W + GHZ key rate

Level 0 returns W only. Levels 1 and 2 return **W + GHZ**, with one common clock and one shared brightness setting.

Keep the supplied `helper.py` next to this notebook and set `SOURCE_DIR` to your existing `.dill` files. Merging, corrections, and robust searches are imported. The source expressions are unchanged. Use your corrected hybrid exports containing both QM-readout and QFC loss.

All times are in microseconds; rates are in bits/s. The calculation retains the previous $11/6$ and factor-3 time bounds, and sums the accepted routes as
$$R_{\rm total}=10^6\frac{\sum_r p_r f_r}{C_{\rm cycle}+P_{\rm accepted}t_{\rm meas}}.$$


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import helper as hp

SOURCE_DIR = Path(".")
POSTPROCESSING = "best"   # "one_way", "ad" (forced), or "best" (optional)
FINAL_POOLING = "route"      # "route" or "class"
C = 0.2                     # km / microsecond
NUM_FREQ = 100
bounds = [(0.0, 0.99), (0.0, 0.3)]  # q and lambda
SEARCH_OPTIONS = dict(n_log=25, n_linear=13, log_decades=6.0,
                      grid_refinements=1, max_starts=8, edge_starts=3,
                      local_tol=1e-8, local_maxfev=600,
                      refinement_rtol=1e-4, warn=False)


## Source expressions

In [2]:
argument_names = ("q", "lambda", "p_r", "p_l")
get_prob_click = hp.load_source_function(
    SOURCE_DIR / "hybrid_prob_click.dill", argument_names)
get_ion_dm = hp.load_source_function(
    SOURCE_DIR / "hybrid_ion_dm.dill", argument_names)
get_prob_load = hp.load_source_function(
    SOURCE_DIR / "hybrid_prob_load.dill", argument_names)


## Overall key rate

In [3]:
def get_key_rate_components(vars, d, p_l_val, num_level=2, *,
                            postprocessing=None, pooling=None, return_states=False):
    """Total, W, and GHZ contributions at one common source setting."""
    mode = POSTPROCESSING if postprocessing is None else postprocessing
    pool = FINAL_POOLING if pooling is None else pooling
    q_val, lam_val = map(float, vars)
    d_ES = d / (2 ** (num_level + 1))
    t_signal = 2.0 * d_ES / C
    p_r_val = hp.get_loss_ratio(d_ES)
    args = (q_val, lam_val, p_r_val, p_l_val)
    p_single = hp.probability(get_prob_click(*args), "single-mode click")
    p_click = hp.multiplexed_click_probability(p_single, NUM_FREQ)
    p_load = hp.probability(get_prob_load(*args), "loading") if p_click > 0 else 0.0
    p_EL = p_click * p_load
    # attempt_time = 1.0 + t_signal + 100.0 + t_signal  # fixed-slot policy
    t_QM_try = 1.0 + t_signal
    t_load = 100.0 + t_signal
    attempt_time = t_QM_try + p_click * t_load # No need to try load if the remote generation fails.
    elementary = dict(prob_click_per_frequency=p_single, prob_click=p_click,
                      prob_load=p_load, prob_EL=p_EL, attempt_time_us=attempt_time,
                      distance_to_swapper_km=d_ES, remote_loss=float(p_r_val),
                      num_frequency_modes=NUM_FREQ, timing_policy="conditional_loading")
    if p_EL == 0.0:  # No herald: its conditional state is undefined.
        parts = hp.empty_rates(num_level, mode, pool)
    else:
        parts = hp.overall_network_rates(
            get_ion_dm(*args), p_EL / attempt_time,
            t_merge_1=2.0 * (200.0 + 2.0 * t_signal),
            t_merge_2=2.0 * (200.0 + 4.0 * t_signal),
            num_level=num_level, t_meas=100.0,
            postprocessing=mode, pooling=pool, return_states=return_states)
    parts.update(distance_km=d, q=q_val, p_l=p_l_val, elementary=elementary)
    parts["lambda"] = lam_val
    return parts


def get_key_rate(vars, d, p_l_val, num_level=2):
    """W-only at level 0; overall W + GHZ at levels 1 and 2."""
    return get_key_rate_components(vars, d, p_l_val, num_level)["total_bps"]


## Evaluate or optimize

`get_key_rate([0.05, 0.02], d=100, p_l_val=0.1, num_level=2)` returns the total in bits/s. Use `get_key_rate_components(...)` for `W_bps`, `GHZ_bps`, and `total_bps`.

`optimize_rate(d=100, p_l_val=0.1, num_level=2)` returns the best evaluated parameters in `.x` and the negative total rate in `.fun`.


In [4]:
def optimize_rate(d, p_l_val, num_level=2, *, previous=None, **options):
    """Maximize the total, not the two components independently."""
    settings = {**SEARCH_OPTIONS, **options}
    return hp.optimize_hybrid(get_key_rate, d, p_l_val, num_level,
                              bounds=bounds, previous_x=previous, **settings)


def run_overall_sweeps(distances=None, losses=(0.01, 0.1, 0.5), levels=(0, 1, 2),
                       *, output_dir=None, search_options=None, verbose=True):
    def optimize(d, loss, level, previous):
        return optimize_rate(d, loss, level, previous=previous,
                             **(search_options or {}))
    return hp.run_rate_sweeps(
        get_key_rate_components, optimize, "hybrid",
        distances=distances, losses=losses, levels=levels,
        output_dir=output_dir, postprocessing=POSTPROCESSING,
        pooling=FINAL_POOLING, verbose=verbose)


## Optimize and save

Edit the grid below, then run this cell. One CSV per local loss is saved in `overall_rates_<postprocessing>_<pooling>/`.

`keyrate_bps_level_N` is already **W + GHZ**. The W/GHZ component columns use the same optimized parameters. Do not add a separate GHZ rate to the total again.

In [5]:
DISTANCES = np.linspace(0.0, 500.0, 20)
LOSSES = (0.01, 0.1, 0.5)
LEVELS = (0, 1, 2)

tables, optimizer_results, route_rates = run_overall_sweeps(
    distances=DISTANCES, losses=LOSSES, levels=LEVELS)

display(tables[0.1])


hybrid: loss=0.01, level=0, d=0 km, total=228.825 bps
hybrid: loss=0.01, level=0, d=26.3158 km, total=55.5506 bps
hybrid: loss=0.01, level=0, d=52.6316 km, total=24.8381 bps
hybrid: loss=0.01, level=0, d=78.9474 km, total=12.101 bps
hybrid: loss=0.01, level=0, d=105.263 km, total=5.68309 bps
hybrid: loss=0.01, level=0, d=131.579 km, total=2.45822 bps
hybrid: loss=0.01, level=0, d=157.895 km, total=0.977987 bps
hybrid: loss=0.01, level=0, d=184.211 km, total=0.367576 bps
hybrid: loss=0.01, level=0, d=210.526 km, total=0.1346 bps
hybrid: loss=0.01, level=0, d=236.842 km, total=0.0489874 bps
hybrid: loss=0.01, level=0, d=263.158 km, total=0.017883 bps
hybrid: loss=0.01, level=0, d=289.474 km, total=0.00656866 bps
hybrid: loss=0.01, level=0, d=315.789 km, total=0.00242903 bps
hybrid: loss=0.01, level=0, d=342.105 km, total=0.000903939 bps
hybrid: loss=0.01, level=0, d=368.421 km, total=0.000338306 bps
hybrid: loss=0.01, level=0, d=394.737 km, total=0.000127249 bps
hybrid: loss=0.01, level=

,distance_km,keyrate_bps_level_0,keyrate_W_bps_level_0,keyrate_GHZ_bps_level_0,opt_qs_level_0,p_accept_level_0,generation_time_us_level_0,round_time_us_level_0,opt_lambdas_level_0,keyrate_bps_level_1,...,round_time_us_level_1,opt_lambdas_level_1,keyrate_bps_level_2,keyrate_W_bps_level_2,keyrate_GHZ_bps_level_2,opt_qs_level_2,p_accept_level_2,generation_time_us_level_2,round_time_us_level_2,opt_lambdas_level_2
0,0.000000,5.477251e+01,5.477251e+01,0.0,0.342652,1.0,1.625637e+03,1.725637e+03,0.300000,5.638839,...,1.368438e+04,0.300000,0.329853,0.043789,0.286064,0.076675,0.680371,2.491084e+05,2.492084e+05,0.300000
1,26.315789,1.021655e+01,1.021655e+01,0.0,0.360028,1.0,6.507146e+03,6.607146e+03,0.092027,1.274598,...,4.123455e+04,0.053053,0.094118,0.000000,0.094118,0.093058,0.679948,3.995000e+05,3.996000e+05,0.040709
2,52.631579,3.609747e+00,3.609747e+00,0.0,0.366870,1.0,1.478725e+04,1.488725e+04,0.101765,0.478163,...,9.177130e+04,0.049941,0.040869,0.000000,0.040869,0.092998,0.678699,7.817048e+05,7.818048e+05,0.036520
3,78.947368,1.353826e+00,1.353826e+00,0.0,0.370740,1.0,3.321213e+04,3.331213e+04,0.110210,0.213986,...,1.844388e+05,0.048439,0.021000,0.000000,0.021000,0.092866,0.678014,1.387484e+06,1.387584e+06,0.033889
4,105.263158,4.910917e-01,4.910917e-01,0.0,0.372776,1.0,8.181215e+04,8.191215e+04,0.115620,0.104565,...,3.534371e+05,0.047620,0.011952,0.000000,0.011952,0.092754,0.677601,2.303082e+06,2.303182e+06,0.032108
5,131.578947,1.727547e-01,1.727547e-01,0.0,0.373724,1.0,2.190088e+05,2.191088e+05,0.118401,0.053836,...,6.587254e+05,0.047140,0.007281,0.000000,0.007281,0.092670,0.677334,3.642278e+06,3.642378e+06,0.030850
6,157.894737,6.025625e-02,6.025625e-02,0.0,0.374129,1.0,6.108001e+05,6.109001e+05,0.119647,0.030041,...,3.051671e+05,0.077656,0.004653,0.000000,0.004653,0.092607,0.677154,5.555675e+06,5.555775e+06,0.029933
7,184.210526,2.114268e-02,2.114268e-02,0.0,0.374295,1.0,1.720500e+06,1.720600e+06,0.120168,0.017194,...,5.097930e+05,0.078563,0.003080,0.000000,0.003080,0.092561,0.677027,8.242536e+06,8.242636e+06,0.029250
8,210.526316,7.502818e-03,7.502818e-03,0.0,0.374361,1.0,4.825114e+06,4.825214e+06,0.120380,0.009878,...,8.603442e+05,0.079196,0.002094,0.000000,0.002094,0.092526,0.676935,1.196603e+07,1.196613e+07,0.028734
9,236.842105,2.694500e-03,2.694500e-03,0.0,0.374388,1.0,1.340950e+07,1.340960e+07,0.120465,0.005698,...,1.460860e+06,0.079621,0.001454,0.000000,0.001454,0.092500,0.676868,1.707303e+07,1.707313e+07,0.028338
